# BirdCLEF 2026 — perch-label-head + ONNX

- **ONNX**: Perch→ONNX at runtime + head ONNX
- **Local**: 0.4900 (no PP) — baseline with ONNX speedup
- **Datasets**: `birdclef2026-perch-label-head` · `birdclef2026-perch-label-head-onnx`


In [ ]:
import subprocess, sys

# Install TF 2.20 if needed (Kaggle ships 2.19 which breaks Perch v2)
try:
    import tensorflow as tf
    assert tuple(int(x) for x in tf.__version__.split('.')[:2]) >= (2, 20)
except (ImportError, AssertionError):
    import os
    tb = '/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorboard-2.20.0-py3-none-any.whl'
    tf_ = '/kaggle/input/bc26-tensorflow-2-20-0/wheel/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
    for w in [tb, tf_]:
        if os.path.isfile(w):
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', w], check=False)

import tensorflow as tf
print(f'TensorFlow {tf.__version__}')
assert tuple(int(x) for x in tf.__version__.split('.')[:2]) >= (2, 20)
tf.experimental.numpy.experimental_enable_numpy_behavior()

In [ ]:
import glob, os, re, time
from concurrent.futures import ThreadPoolExecutor
import h5py
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf

START          = time.time()
TERMINATE_TIME = START + 5300   # 88 min — leave 2 min buffer for submission write

## Config

In [ ]:
DATA_DIR      = '/kaggle/input/birdclef-2026'
PERCH_DIR     = '/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1'
WEIGHTS_PATH  = '/kaggle/input/birdclef2026-perch-label-head/best_head.weights.h5'

ALPHA         = 0.4   # Perch zero-shot weight

NUM_CLASSES   = 234
HIDDEN_DIM    = 256
SR            = 32_000
CLIP_SAMPLES  = SR * 5   # 5-second clips

# Soundscapes: use test if available, else train
TEST_DIR = os.path.join(DATA_DIR, 'test_soundscapes')
SC_DIR   = TEST_DIR if glob.glob(os.path.join(TEST_DIR, '*.ogg')) else os.path.join(DATA_DIR, 'train_soundscapes')
ogg_files = sorted(glob.glob(os.path.join(SC_DIR, '*.ogg')))
print(f'Soundscapes: {len(ogg_files)} files in {SC_DIR}')

## Species Mapping (Perch → BirdCLEF 234 species)

In [ ]:
bc_labels = pd.read_csv(os.path.join(PERCH_DIR, 'assets/labels.csv'))
bc_labels = (bc_labels.reset_index()
             .rename({'inat2024_fsd50k': 'scientific_name', 'index': 'bc_index'}, axis=1)
             .set_index('scientific_name'))
N_PERCH = len(bc_labels)

taxonomy       = pd.read_csv(os.path.join(DATA_DIR, 'taxonomy.csv'))
sample_sub     = pd.read_csv(os.path.join(DATA_DIR, 'sample_submission.csv'))
primary_labels = sample_sub.columns[1:].tolist()

mapping = taxonomy.join(bc_labels, on='scientific_name', how='left')
mapping.bc_index = mapping.bc_index.fillna(N_PERCH).astype(int)
mapping = mapping[['primary_label', 'bc_index']].set_index('primary_label')

# Index for gathering from padded Perch label output; OOV → N_PERCH → 0 after pad
perch_indices = [int(mapping.loc[pl].iloc[0]) if pl in mapping.index else N_PERCH
                 for pl in primary_labels]
n_covered = sum(1 for i in perch_indices if i < N_PERCH)
print(f'Perch covers {n_covered}/{NUM_CLASSES} target species')

## Load Perch + Label-Head

In [ ]:
# ── ONNX Runtime Setup ─────────────────────────────────────────────────────────
# Converts Perch (perch_v2_cpu) to ONNX at first run, caches to /kaggle/working.
# Falls back to TF if conversion fails. Label-head always uses ONNX (50× faster).
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tf2onnx'], check=False)
import tf2onnx
import onnxruntime as ort
import h5py

PERCH_ONNX_PATH = '/kaggle/working/perch_cpu.onnx'
HEAD_ONNX_PATH  = '/kaggle/input/birdclef2026-perch-label-head-onnx/head_perch-label-head.onnx'

# ── Convert Perch (once) ──────────────────────────────────────────────────────
USE_ONNX_PERCH = False
if not os.path.exists(PERCH_ONNX_PATH):
    print('Converting Perch to ONNX (one-time, ~2-3 min)...')
    try:
        _, _ = tf2onnx.convert.from_saved_model(
            PERCH_DIR, opset=17, output_path=PERCH_ONNX_PATH,
            extra_opset=[{"domain":"com.microsoft","version":1}]
        )
        print(f'Perch ONNX saved ({os.path.getsize(PERCH_ONNX_PATH)//1024//1024} MB)')
        USE_ONNX_PERCH = True
    except Exception as e:
        print(f'Perch ONNX conversion failed (using TF fallback): {e}')
else:
    print(f'Perch ONNX cache found.')
    USE_ONNX_PERCH = True

# ── Load sessions ─────────────────────────────────────────────────────────────
ORT_OPTS = ort.SessionOptions()
ORT_OPTS.intra_op_num_threads = 4

if USE_ONNX_PERCH:
    perch_sess = ort.InferenceSession(PERCH_ONNX_PATH, sess_options=ORT_OPTS,
                                       providers=['CPUExecutionProvider'])
    print(f'Perch ONNX loaded. Inputs: {[i.name for i in perch_sess.get_inputs()]}')

head_sess = ort.InferenceSession(HEAD_ONNX_PATH, sess_options=ORT_OPTS,
                                  providers=['CPUExecutionProvider'])
print(f'Head ONNX loaded (perch label-head 234->256->234)')

# ── TF Perch fallback ─────────────────────────────────────────────────────────
if not USE_ONNX_PERCH:
    print('Loading Perch via TensorFlow (fallback)...')
    perch_model_tf = tf.saved_model.load(PERCH_DIR)
    sig_tf = perch_model_tf.signatures['serving_default']
    print(f'TF Perch loaded.')

# ── Numpy helpers ─────────────────────────────────────────────────────────────
def _sigmoid(x): return 1.0 / (1.0 + np.exp(-np.clip(x, -88, 88)))

def _infer_clips(clips_np):
    """Run Perch + label-head. Uses ONNX if available, else TF."""
    if USE_ONNX_PERCH:
        perch_out = perch_sess.run(['label'], {'inputs': clips_np})[0]  # (N, N_PERCH)
    else:
        out = sig_tf(inputs=tf.constant(clips_np))
        perch_out = out['label'].numpy()                                  # (N, N_PERCH)

    label_pad = np.pad(perch_out, [[0,0],[0,1]])                          # (N, N_PERCH+1)
    features  = label_pad[:, np.array(perch_indices_list)]                # (N, 234)
    head_logits = head_sess.run(None, {'input': features.astype(np.float32)})[0]
    return ALPHA * _sigmoid(features) + (1-ALPHA) * _sigmoid(head_logits)

print(f"Inference mode: {'ONNX (Perch + Head)' if USE_ONNX_PERCH else 'ONNX Head + TF Perch'}")


In [ ]:
# Store perch_indices as plain Python list for ONNX numpy indexing
perch_indices_list = perch_indices


## Inference with ThreadPoolExecutor

In [ ]:
def predict_file(ogg_path):
    """ONNX-accelerated inference."""
    ss_id   = re.sub(r'\.ogg$', '', os.path.basename(ogg_path), flags=re.IGNORECASE)
    row_ids = [f'{ss_id}_{n}' for n in range(5, 65, 5)]
    zeros   = np.zeros((12, NUM_CLASSES), dtype=np.float32)
    if time.time() > TERMINATE_TIME:
        return row_ids, zeros
    try:
        audio, _ = librosa.load(ogg_path, sr=SR, mono=True)
        audio = audio.astype(np.float32)
        remainder = len(audio) % CLIP_SAMPLES
        if remainder:
            audio = np.pad(audio, (0, CLIP_SAMPLES - remainder))
        clips = audio.reshape(-1, CLIP_SAMPLES)
        n_clips = min(len(clips), 12)
        preds = _infer_clips(clips[:n_clips])
        if n_clips < 12:
            preds = np.vstack([preds, np.zeros((12-n_clips, NUM_CLASSES), dtype=np.float32)])
        return row_ids, preds
    except Exception as e:
        print(f'ERROR {os.path.basename(ogg_path)}: {e}')
        return row_ids, zeros


In [ ]:
all_row_ids, all_preds = [], []

with ThreadPoolExecutor(max_workers=4) as executor:
    for k, (row_ids, preds) in enumerate(executor.map(predict_file, ogg_files)):
        all_row_ids.extend(row_ids)
        all_preds.append(preds)
        if k % 100 == 0:
            elapsed = (time.time() - START) / 60
            print(f'[{k+1}/{len(ogg_files)}] {elapsed:.1f} min elapsed')

print(f'Done: {len(all_row_ids)} rows in {(time.time()-START)/60:.1f} min')

## Build Submission

In [ ]:
# ── Post-processing: median(size=3) — +0.0034 local ROC-AUC ──────────────────
from scipy.ndimage import median_filter

sc_groups = {}
for i, rid in enumerate(all_row_ids):
    sc_groups.setdefault(rid.rsplit("_",1)[0], []).append(i)
for s in sc_groups:
    sc_groups[s].sort(key=lambda i: int(all_row_ids[i].rsplit("_",1)[1]))

def temporal_median(preds, size=3):
    smoothed = preds.copy()
    for idxs in sc_groups.values():
        if len(idxs) > 1:
            smoothed[idxs] = median_filter(preds[idxs], size=(size, 1), mode="nearest")
    return np.clip(smoothed, 0.0, 1.0)

predictions = np.concatenate(all_preds, axis=0)
predictions = temporal_median(predictions, size=3)
print(f"PP done. range [{predictions.min():.4f}, {predictions.max():.4f}]")


In [ ]:
submission  = pd.DataFrame(predictions, columns=primary_labels)
submission.insert(0, 'row_id', all_row_ids)
submission  = sample_sub[['row_id']].merge(submission, on='row_id', how='left').fillna(0.0)
submission.to_csv('submission.csv', index=False)
print(f'submission.csv: {submission.shape[0]} rows x {len(primary_labels)} species')
print(f'Total elapsed: {(time.time()-START)/60:.1f} min')
submission.head(5)
